In [ ]:
import os
import sys
import logging
import pandas as pd
import random
import shutil

from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler

# Add project root to system path (for relative imports to work)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src import config

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

logger.info("Project root: %s", project_root)


In [ ]:
#  Imports & Configuration

# User parameters

if config.USE_SAMPLED_TRAIN_DATASET:
    INPUT_DIR    = config.SAMPLED_TRAIN_DATASET_DIR         # path of sampled training dataset
else:
    INPUT_DIR    = config.TRAIN_DATASET_DIR                  # path of training dataset

PREPROCESSED_OUTPUT_DIR   = config.PREPROCESSED_DATASET_DIR
TARGET_SIZE  = (640, 640)                          # (height, width)
EXTS         = [".jpg", ".png", ".tif", ".tiff"]   # supported extensions

# CLAHE & denoise settings
CLAHE_CFG    = {"clip_limit": 2.0, "grid_size": (8, 8)}
DENOISE_CFG  = {"h": 10, "template_size": 7, "search_size": 21}

# Ensure output directory exists
os.makedirs(PREPROCESSED_OUTPUT_DIR, exist_ok=True)

logger.info("Input dir: %s", INPUT_DIR)
logger.info("Preprocessed output dir: %s", PREPROCESSED_OUTPUT_DIR)


In [ ]:
from src.preprocess_utils import (
    ensure_project_root_in_sys_path,
    build_positive_samples,
    collect_all_images,
    balance_paths,
    process_images,
)

data_root    = config.TRAIN_DATASET_HPC_DIR
labels_csv   = config.TRAIN_LABELS_PATH 

output_base         = os.path.join(config.WORKSPACE_ROOT, "data", "yolo")
output_images_train = os.path.join(output_base, "images", "train")
output_images_val   = os.path.join(output_base, "images", "val")
output_labels_train = os.path.join(output_base, "labels", "train")
output_labels_val   = os.path.join(output_base, "labels", "val")

box_size    = 32

total_cap     = 2000
neg_cap       = total_cap // 2   # 1000 negatives
pos_cap       = total_cap - neg_cap  # 1000 positives

pipeline = Pipeline([
    ("undersample", RandomUnderSampler(sampling_strategy={0: neg_cap}, random_state=42)),
    ("oversample",  RandomOverSampler (sampling_strategy={1: pos_cap}, random_state=42)),
])

for path in (output_images_train, output_images_val, output_labels_train, output_labels_val):
    os.makedirs(path, exist_ok=True)

labels_df = pd.read_csv(labels_csv)

positive_samples = build_positive_samples(labels_df, data_root)

all_images = collect_all_images(data_root, exts=[".jpg", ".png", ".tif", ".tiff"]) 

df = pd.DataFrame({
    "path": all_images,
    "label": [1 if p in positive_samples else 0 for p in all_images]
})

print(df)

X_res, y_res = balance_paths(df, pipeline)
balanced_paths = X_res["path"].tolist()
balanced_images = X_res["path"].tolist()

print(X_res)
print(y_res)

random.seed(42)
random.shuffle(balanced_images)
split_idx    = int(len(balanced_images) * 0.8)
train_images = balanced_images[:split_idx]
val_images   = balanced_images[split_idx:]

process_images(train_images, positive_samples, output_images_train, output_labels_train, box_size)
process_images(val_images, positive_samples, output_images_val, output_labels_val, box_size)


Convert raw sample data to yolo format

[Object Detection dataset overview](https://docs.ultralytics.com/datasets/detect/)

Data augmentation for training data using albumentations liblary.